# Constrained Multi-Objective ASD Optimization

This notebook demonstrates the Stage 2 benchmark for the educational ASD virtual laboratory. The simulator is a toy model for agent design, optimization, benchmarking, and failure analysis; it does not predict real HfO2/MoS2 chemistry.

## Why Selectivity Alone Is Insufficient

A condition can have high selectivity when both surfaces barely grow. That is not useful if the growth area never reaches a practical thickness. Stage 2 therefore treats useful GA growth, suppressed NGA growth, and process time as objectives, while selectivity is a constraint together with minimum GA thickness and maximum NGA thickness.

## Mixed Variables

The Stage 2 search space contains two continuous variables, precursor dose and temperature, plus one integer variable, cycle count. The MOBO method handles cycle count by enumerating permitted integer values and optimizing the continuous variables conditional on each value.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from asd_agent.bo.stage2_analysis import generate_stage2_analysis_outputs  # noqa: E402
from asd_agent.bo.stage2_benchmark import (  # noqa: E402
    load_stage2_benchmark_profile,
    profile_configs,
    run_stage2_benchmark,
    stage2_summary_rows,
)

profile = load_stage2_benchmark_profile("smoke").model_copy(
    update={"scenarios": ["inherent_selectivity"], "budget": 3}
)
profile

In [ ]:
results = run_stage2_benchmark(profile)
rows = stage2_summary_rows(results)
rows

## Pareto Fronts And Hypervolume

A Pareto front contains tested conditions where no other tested condition is better in all objectives. Hypervolume summarizes how much objective space is dominated by feasible tested points relative to a configured reference point. BO-07 uses area under the feasible hypervolume trajectory as the primary endpoint under a fixed experiment budget.

In [ ]:
output_dir = PROJECT_ROOT / "results" / "notebook_stage2_mobo"
paths = generate_stage2_analysis_outputs(results, output_dir, configs=profile_configs(profile))
[path.relative_to(PROJECT_ROOT) for path in paths[:8]]

In [ ]:
import pandas as pd

summary = pd.DataFrame(rows)
summary[
    [
        "method",
        "status",
        "hypervolume_auc",
        "final_hypervolume",
        "hypervolume_regret",
        "experiments_to_first_feasible",
        "constraint_violation_count",
    ]
]